<a href="https://colab.research.google.com/github/haida-ishtiaq/FlyRankAI-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haida-ishtiaq/FlyRankAI-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item (`content_id`), pseudonymized and belonging to one client (`content_id`). This is the small **starter dataset**, `data/raw/content_refresh_anonymized.csv` — 30,000 rows x 44 columns, one row per content item, 32 clients, per `flyrank-data/SKILL.md`. This is a flat CSV, not the warehouse — there is no `report_date`, `dim_content`, `dim_clients`, or `fact_content_daily_performance` here; those only exist in the separate, gated warehouse release (`hf://datasets/FlyRank/internship-warehouse`), which this notebook does not touch.

**Time window:** there is no explicit calendar date column. Every metric is a trailing window as of an implicit snapshot: `impressions_90d` (last 90 days), `impressions_last_30d` vs `impressions_prev_30d` (most recent 30 days vs. the 30 before that), `content_age_days`, `days_since_last_update`. So every claim below is scoped to "as of this snapshot," not to an absolute date range.

In [1]:
import os
import pandas as pd

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    os.chdir("/content")
    if not os.path.isdir("FlyRankAI-ML-Internship"):
        import subprocess
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/haida-ishtiaq/FlyRankAI-ML-Internship"
        ], check=True)
    os.chdir("FlyRankAI-ML-Internship")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns), "(SKILL.md says 44 — check this matches)")
print("Unique content_id:", df["content_id"].nunique())
print("Duplicate content_id rows (grain probe, should be 0):", df["content_id"].duplicated().sum())
print("Unique client_id:", df["client_id"].nunique(), "(SKILL.md says 32)")

date_like_cols = [c for c in df.columns if "date" in c.lower()]
print("Columns with 'date' in the name (expect none in the starter CSV):", date_like_cols)

window_cols = sorted([c for c in df.columns if "90d" in c or "30d" in c or "age_days" in c])
print("Relative time-window columns found:", window_cols)

Rows: 30000
Columns: 44 (SKILL.md says 44 — check this matches)
Unique content_id: 30000
Duplicate content_id rows (grain probe, should be 0): 0
Unique client_id: 32 (SKILL.md says 32)
Columns with 'date' in the name (expect none in the starter CSV): ['days_since_last_update']
Relative time-window columns found: ['ai_sessions_90d', 'clicks_90d', 'clicks_last_30d', 'clicks_prev_30d', 'content_age_days', 'engaged_sessions_90d', 'impressions_90d', 'impressions_last_30d', 'impressions_prev_30d', 'pageviews_90d', 'scroll_events_90d', 'sessions_90d', 'sessions_last_30d', 'sessions_prev_30d', 'users_90d']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature** (knowable before the prediction moment, safe to use): `impressions_90d`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `days_since_last_update`, `content_age_days`, `word_count`, `freshness_tier`, `days_with_impressions`.

- Note on units, per `flyrank-data/SKILL.md`: `ctr`, `engagement_rate`, `scroll_rate`, and `ai_traffic_pct` are already ×100 percentages (`ctr = 0.76` means 0.76%, not 76%). Any threshold or ratio built on these needs to respect that scale.
- `avg_position == 0` means **no data**, not "rank zero" — 1,205 rows per the skill file. This is treated as missing, verified below.

**Label / proxy** (never a feature): `trend_direction`, `trend_pct`. Per the skill file's "label trap": `is_declining_label` (if present) is derived from `trend_direction`, which is itself computed from `trend_pct` — so both `trend_direction` and `trend_pct` are excluded from any feature set, not just the obviously-named label column.

**Context** (for grouping/joining/reading only, never fed to a model): `content_id`, `client_id` (IDs — pseudonyms, used only for grouped train/test splits per the skill file, never as predictive signal), `content_type`, `main_intent`, `provider_used`, `model_used`.

**Excluded, with why**:
- `client_id`, `content_id` when printing any sample rows — excluded from *printed output* specifically to avoid exposing per-row/per-client identifiers, per the self-check rule ("no client names, URLs, or private queries"). They remain valid *context* columns for internal grouping/splitting, just not for display or as features.
- `impressions_last_30d`, `impressions_prev_30d` — excluded from the feature set because they can directly derive a decline label (`impressions_last_30d < impressions_prev_30d`), which would leak label information into the features. Verified as a leakage risk in Section 3.

In [2]:


features = ["impressions_90d", "ctr", "avg_position", "engagement_rate",
            "scroll_rate", "days_since_last_update", "content_age_days",
            "word_count", "freshness_tier", "days_with_impressions"]
label_cols = ["trend_direction", "trend_pct"]
context_cols = ["content_id", "client_id", "content_type", "main_intent", "provider_used", "model_used"]
excluded_cols = ["impressions_last_30d", "impressions_prev_30d"]

all_named = set(features + label_cols + context_cols + excluded_cols)
unsorted_cols = [c for c in df.columns if c not in all_named]
print("Columns not yet sorted into a bucket (review before final submission):")
print(unsorted_cols)

print("\navg_position == 0 count (should be ~1,205 per the data dictionary):")
print((df["avg_position"] == 0).sum())

print("\nRate-column scale sanity check (ctr, engagement_rate, scroll_rate should look like small percentages, not fractions or raw counts):")
print(df[["ctr", "engagement_rate", "scroll_rate"]].describe().T[["mean", "min", "max"]])

Columns not yet sorted into a bucket (review before final submission):
['search_volume', 'competition', 'competition_level', 'cpc', 'char_count', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_sessions', 'clicks_last_30d', 'sessions_last_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'age_tier', 'age_tier_order', 'word_count_tier', 'char_count_tier', 'ai_traffic_pct', 'impression_tier', 'position_tier']

avg_position == 0 count (should be ~1,205 per the data dictionary):
1205

Rate-column scale sanity check (ctr, engagement_rate, scroll_rate should look like small percentages, not fractions or raw counts):
                      mean  min    max
ctr               0.510733  0.0  100.0
engagement_rate   2.534520  0.0  100.0
scroll_rate      18.212921  0.0  300.0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Per `flyrank-data/SKILL.md`, missingness follows `content_type` — a blind `fillna(0)` would inject a fake category signal. So the check here is not just "how many nulls per column" but "does the null rate differ by `content_type`," which tells us whether we need `has_*` flags instead of imputing.

In [3]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

features = ["impressions_90d", "ctr", "avg_position", "engagement_rate",
            "scroll_rate", "days_since_last_update", "content_age_days",
            "word_count", "freshness_tier", "days_with_impressions"]

# --- Query 1: overall missingness in the planned feature set ---
print("=== Query 1: Missing values in feature set (overall) ===")
print(df[features].isnull().sum())

# --- Query 2: missingness BY content_type, per the skill's warning ---
print("\n=== Query 2: Missingness by content_type (checking for a patterned gap) ===")
null_rate_by_type = df.groupby("content_type")[features].apply(lambda g: g.isnull().mean())
print(null_rate_by_type.round(3))
print("\nIf any content_type shows a much higher null rate than others for a given column,")
print("that column needs a has_<col> flag instead of fillna(0) for that feature.")

# --- Query 3: window consistency check ---
print("\n=== Query 3: Window consistency ===")
overlap_ok = (df["impressions_last_30d"] + df["impressions_prev_30d"] <= df["impressions_90d"]).mean()
print(f"Share of rows where last_30d + prev_30d <= impressions_90d: {overlap_ok:.1%}")

# --- Query 4: leakage check (honest model vs. leaky model) ---
print("\n=== Query 4: Leakage check ===")
work = df.copy()
work["avg_position"] = work["avg_position"].replace(0, np.nan)
work["avg_position"] = work["avg_position"].fillna(work["avg_position"].median())
y = (work["trend_direction"] == "down").astype(int)

numeric_features = [c for c in features if pd.api.types.is_numeric_dtype(work[c])]
X_honest_raw = work[numeric_features].fillna(0)
X_honest = StandardScaler().fit_transform(X_honest_raw)
clf_honest = LogisticRegression(max_iter=2000, random_state=42)
clf_honest.fit(X_honest, y)
auc_honest = roc_auc_score(y, clf_honest.predict_proba(X_honest)[:, 1])
print(f"Honest model ROC-AUC (excluded cols left out): {auc_honest:.4f}")

X_leaky_raw = X_honest_raw.copy()
X_leaky_raw["impressions_last_30d"] = work["impressions_last_30d"]
X_leaky = StandardScaler().fit_transform(X_leaky_raw)
clf_leaky = LogisticRegression(max_iter=2000, random_state=42)
clf_leaky.fit(X_leaky, y)
auc_leaky = roc_auc_score(y, clf_leaky.predict_proba(X_leaky)[:, 1])
print(f"Leaky model ROC-AUC (impressions_last_30d added back in): {auc_leaky:.4f}")
print(f"AUC jump from adding the excluded column: {auc_leaky - auc_honest:+.4f}")
print("A large jump here confirms impressions_last_30d is correctly excluded as a leakage risk.")

=== Query 1: Missing values in feature set (overall) ===
impressions_90d              0
ctr                          0
avg_position                 0
engagement_rate              0
scroll_rate                125
days_since_last_update       0
content_age_days             0
word_count                7699
freshness_tier               0
days_with_impressions        0
dtype: int64

=== Query 2: Missingness by content_type (checking for a patterned gap) ===
                    impressions_90d  ctr  avg_position  engagement_rate  \
content_type                                                              
comparison article              0.0  0.0           0.0              0.0   
feedly article                  0.0  0.0           0.0              0.0   
keyword article                 0.0  0.0           0.0              0.0   

                    scroll_rate  days_since_last_update  content_age_days  \
content_type                                                                
comparison ar

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Based only on what Section 3's queries show, plus what `flyrank-data/SKILL.md` names directly:

- **This is the small starter slice, not the warehouse.** 30,000 rows across 32 clients is a convenience sample for developing logic — it is not the ~79M-row warehouse (`fact_content_daily_performance`), which has real per-client history depth differences (`dim_clients.gsc_data_start`), a real `ga4_data_available` flag, and a sealed final-month test partition. None of those warehouse-specific concepts (GA4 availability flags, per-client start dates, sealed test months) apply to this CSV — any contract or limitation written for the warehouse should not be copy-pasted onto this dataset.
- **No calendar dates.** Confirmed in Section 1 — every metric is a trailing window "as of" an unstated snapshot, not tied to an absolute date.
- **No causal information.** `trend_direction` is an observed outcome, not evidence of *why* a page moved. Any ranking built on this data is directional/decision-support only.
- **No refresh-history flag.** Nothing indicates whether a page was already manually refreshed during the observed window — a real confound for any later claim like "refreshing improves ranking."
- **Missingness is patterned, not random** (Section 3, Query 2) — it follows `content_type`, so any modeling work downstream should use `has_*` flags for affected columns rather than a blind `fillna(0)`, which would otherwise inject a fake category signal per the skill file's warning.

## 5. Output

*What the analysis hands to the human, in one sentence.*

This analysis hands the modeling stage (`w05_model.ipynb`) a clean, page-level feature table — 10 verified features with documented missingness patterns, `trend_direction`/`trend_pct` withheld as the label source, and `impressions_last_30d`/`impressions_prev_30d` withheld as a confirmed leakage risk — ready to predict `is_declining` without re-deriving any of this groundwork.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.